In [1]:
import sys
import numpy as np
import pandas as pd
import re
from nltk.corpus import stopwords, wordnet
from nltk.stem import WordNetLemmatizer
from nltk import pos_tag
import plotly.express as px
import plotly.graph_objects as go
import itertools
from IPython.display import display
import scipy
import multiprocess
from pathlib import Path
project_root = Path(__file__).resolve().parents[0] if '__file__' in globals() else Path().resolve().parents[0]
sys.path.insert(0, str(project_root))
pd.set_option('display.max_columns', None)

# Preprocessing

In [2]:
from tomato.utils import tomato_data_path

# load stratified downsampled data
df = pd.read_csv(tomato_data_path() / 'df_strat800.csv')

In [3]:
# initialize list of english stop words
stopword_list = stopwords.words('english')
# initialize lemmatizer
lem = WordNetLemmatizer()

# converts default pos_tagger tag to lemmatizer friendly tag 
def get_wordnet_pos(treebank_tag):

    if treebank_tag.startswith('J'):
        return wordnet.ADJ
    elif treebank_tag.startswith('V'):
        return wordnet.VERB
    elif treebank_tag.startswith('N'):
        return wordnet.NOUN
    elif treebank_tag.startswith('R'):
        return wordnet.ADV
    else:
        return wordnet.NOUN

# preprocessing function 
def preprocess_string(string):
    # lowercasing
    clean_string = string.lower()
    # remove special characters
    clean_string = re.sub(r'[^a-z]', ' ', clean_string)
    # standardize spacing
    clean_string = re.sub(r'\s+', r' ', clean_string)
    # remove stop words
    clean_string = ' '.join(
        [word for word in clean_string.split(' ') if word not in stopword_list]
    )
    # part-of-speech tagging
    word_pos_zip = [(i,get_wordnet_pos(j)) for (i,j) in pos_tag(clean_string.split(' '))]
    # lemmatization
    clean_string = ' '.join(
        [lem.lemmatize(word, pos) for (word,pos) in word_pos_zip]
    )
    return clean_string

In [4]:
# preprocess messy review strings into clean strings, ready to be tokenized
df['clean_review_content'] = df['review_content'].map(lambda x: 
    preprocess_string(x)
)

# Embedding

### TF-IDF

In [11]:
from sklearn.feature_extraction.text import TfidfVectorizer
tfidf = TfidfVectorizer(min_df=2, ngram_range=(1,3))
embedding_tfidf = tfidf.fit_transform(
    df.clean_review_content
).todense()
features_tfidf = tfidf.get_feature_names_out()
print(embedding_tfidf.shape)
print(features_tfidf)
# persist embedding
np.save(tomato_data_path() / 'embedding_tfidf_strat800.npy', embedding_tfidf)
np.save(tomato_data_path() / 'features_tfidf_strat800.npy', features_tfidf)

(800, 1845)
['ability' 'able' 'able take' ... 'young viewer' 'young viewer thoughtful'
 'zellweger']


### Binary Bag-of-Words

In [12]:
from sklearn.feature_extraction.text import CountVectorizer
bow = CountVectorizer(binary=True, min_df=2)
embedding_bow = bow.fit_transform(
    df.clean_review_content
).todense()
features_bow = bow.get_feature_names_out()
print(embedding_bow.shape)
print(features_bow)
# persist embedding
np.save(tomato_data_path() / 'embedding_bow_strat800.npy', embedding_bow)
np.save(tomato_data_path() / 'features_bow_strat800.npy', features_bow)

(800, 1120)
['ability' 'able' 'absolutely' ... 'york' 'young' 'zellweger']
